# LGFDR anomaly detection — Swin-B backbone on industrial wood-panel images

Unsupervised anomaly detection with **Local-Global Feature Dual Reconstruction (LGFDR)**:

1. A pretrained backbone (`timm`, `features_only=True`) extracts the first two feature stages, which are upsampled,
   concatenated and average-pooled into a feature map **E**.
2. Gaussian noise is injected into **E** during training only.
3. Two convolutional auto-encoders reconstruct the features: a **local** tower (1x1 convs) and a **global**
   tower (strided 3x3 convs + bilinear upsampling).
4. A **Selective Reconstruction Head** (channel-wise softmax over the two towers) fuses them into **U**.
5. The per-pixel L2 distance `||U - E||` is the anomaly map; its pooled maximum is the image-level score.

The model is trained only on defect-free images. Training first fits the reconstruction heads with the backbone
frozen and unfreezes the backbone (lower learning rate) when the loss plateaus; it stops on loss patience.

**Data:** a proprietary industrial dataset of wood-panel surface images (normal vs. defective, JSON label per image,
several panel colours) from an industrial project. The data cannot be shared; the notebook documents the method
and the numbers it produced. Product identifiers have been anonymised.

In [ ]:
%pip install -q torch torchvision timm scikit-learn matplotlib tqdm

In [ ]:
import os

# Root folder of the dataset(s) used in this notebook.
# Set the DATA_ROOT environment variable or edit the default below.
# On Google Colab you can first mount Drive with:
#   from google.colab import drive; drive.mount('/content/drive')
DATA_ROOT = os.environ.get("DATA_ROOT", "./data")

In [ ]:
images_size_globally = 224
model_name_globally ='swin_base_patch4_window7_224'

In [ ]:
# Updated SwinFeatureExtractor class
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm

class SwinFeatureExtractor(nn.Module):
    # Removed the freeze_bn parameter and logic from here
    def __init__(self, model_name=model_name_globally, pretrained=True, pool_kernel=8):
        super().__init__()
        # Ensure pretrained is True for using pre-trained weights
        self.model = timm.create_model(model_name, pretrained=pretrained, features_only=True)
        self.pool_kernel = pool_kernel



        # The freeze_extractor flag in the LGFDRModel will now control
        # requires_grad for ALL parameters in self.model, including BN weights/biases.
        # The train/eval mode of the LGFDRModel will control BN running stats updates.


    def forward(self, x):
        # The parent model's train/eval mode will now control the behavior of BatchNorm layers.
        # No explicit eval() call needed here just for frozen BN anymore.
        features = self.model(x)  # List of feature maps from stages
        features = [f.permute(0, 3, 1, 2) if f.ndim == 4 else f for f in features]
        if len(features) < 2:
             raise ValueError(f"Feature extractor '{self.model.__class__.__name__}' for model '{self.model.pretrained_cfg['architecture']}' returned less than 2 feature stages ({len(features)}). Cannot proceed.")
        feat1, feat2 = features[:2]  # [(B, C1, H1, W1), (B, C2, H2, W2)]
        # Upsample both to largest (finest) spatial size
        target_size = feat1.shape[-2:]
        # Check if spatial sizes are already the same to avoid unnecessary interpolation
        if feat1.shape[-2:] != feat2.shape[-2:]:
            feat2_up = F.interpolate(feat2, size=target_size, mode='bilinear', align_corners=False)
        else:
            feat2_up = feat2 # No need to interpolate if sizes match

        # Concatenate along channel
        # Ensure number of dimensions match before concatenation
        if feat1.ndim != 4 or feat2_up.ndim != 4:
             raise ValueError(f"Feature stages are not 4D after processing: feat1={feat1.shape}, feat2_up={feat2_up.shape}")

        feat_cat = torch.cat([feat1, feat2_up], dim=1)
        # Average Pooling for false positive reduction (kernel=8)
        feat_pooled = F.avg_pool2d(feat_cat, kernel_size=self.pool_kernel, stride=1, padding=self.pool_kernel//2)
        return feat_pooled  # E: (B, C, H, W)

# You would then instantiate the LGFDRModel passing freeze_extractor=True
# The LGFDRModel's freeze_extractor=True will now set requires_grad=False for all extractor params, *including* BN weights/biases.
# The BN running stats will update during model.train() because you removed the module.eval() call in the extractor's init.
# Example:
# model = LGFDRModel(freeze_extractor=True).to(device) # Use freeze_extractor=True here

In [ ]:
# 2. Noise Injection: Add Gaussian noise during training only
def inject_noise(E, is_train):
    # E: (B, C, H, W)
    if is_train:
        noise = torch.randn_like(E)
        return E + noise
    else:
        return E

In [ ]:
# 3. Dual-Tower CAE Reconstruction Network (IMPROVED)
import torch
import torch.nn as nn
import torch.nn.functional as F

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, kernel_size=3, stride=1, padding=1, use_bn=True, use_relu=True):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_c) if use_bn else nn.Identity()
        self.relu = nn.ReLU(inplace=True) if use_relu else nn.Identity()
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))

class LocalCAE(nn.Module):
    # Uses Conv1x1 (standard choice)
    def __init__(self, c):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(c, c, kernel_size=1, padding=0),
            ConvBlock(c, c, kernel_size=1, padding=0),
            ConvBlock(c, c, kernel_size=1, padding=0)
        )
        self.decoder = nn.Sequential(
            ConvBlock(c, c, kernel_size=1, padding=0),
            ConvBlock(c, c, kernel_size=1, padding=0),
            ConvBlock(c, c, kernel_size=1, padding=0, use_relu=False)  # No ReLU in the last block!
        )
        # Optional: output normalization (uncomment if needed)
        # self.out_norm = nn.BatchNorm2d(c)
    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        # out = self.out_norm(out)  # Uncomment if using output normalization
        return out

class GlobalCAE(nn.Module):
    # Uses Conv3x3, stride-2 for down, bilinear up (standard choice)
    def __init__(self, c):
        super().__init__()
        self.encoder = nn.Sequential(
            ConvBlock(c, c, kernel_size=3, stride=2, padding=1),
            ConvBlock(c, c, kernel_size=3, stride=2, padding=1),
            ConvBlock(c, c, kernel_size=3, stride=1, padding=1)
        )
        self.decoder = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBlock(c, c, kernel_size=3, stride=1, padding=1),
            nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
            ConvBlock(c, c, kernel_size=3, stride=1, padding=1),
            ConvBlock(c, c, kernel_size=3, stride=1, padding=1, use_relu=False)  # No ReLU in last block!
        )
        # Optional: output normalization (uncomment if needed)
        # self.out_norm = nn.BatchNorm2d(c)
    def forward(self, x):
        z = self.encoder(x)
        out = self.decoder(z)
        # out = self.out_norm(out)  # Uncomment if using output normalization
        return out

# 4. Selective Reconstruction Head (SRH)
class SRH(nn.Module):
    def __init__(self, c, latent_dim=18):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2*c, latent_dim),
            nn.ReLU(inplace=True),
            nn.Linear(latent_dim, 2*c)
        )
    def forward(self, L, G):
        # L, G: (B, C, H, W)
        l = L.mean(dim=[2, 3])  # GAP: (B, C)
        g = G.mean(dim=[2, 3])  # (B, C)
        s = torch.cat([l, g], dim=1)  # (B, 2C)
        u = self.mlp(s)  # (B, 2C)
        ul, ug = torch.chunk(u, 2, dim=1)  # (B, C), (B, C)
        # Softmax across local/global for each channel
        w = torch.stack([ul, ug], dim=2)  # (B, C, 2)
        w_soft = torch.softmax(w, dim=2)
        wL = w_soft[:,:,0].unsqueeze(2).unsqueeze(3)  # (B, C, 1, 1)
        wG = w_soft[:,:,1].unsqueeze(2).unsqueeze(3)  # (B, C, 1, 1)
        U = wL * L + wG * G  # (B, C, H, W)
        return U

In [ ]:
# 5. Anomaly Scoring (Pixel-level and Image-level)
def anomaly_maps(U, E, input_size=(images_size_globally, images_size_globally), pool_kernel=32):
    # U, E: (B, C, H, W)
    # Per-pixel L2 norm over channels
    d = torch.norm(U - E, p=2, dim=1)  # (B, H, W)
    # Upsample to input image size
    D = F.interpolate(d.unsqueeze(1), size=input_size, mode='bilinear', align_corners=False).squeeze(1)  # (B, H0, W0)
    # Average pool anomaly map, then max per image (image-level score)
    D_pool = F.avg_pool2d(D, kernel_size=pool_kernel, stride=1, padding=pool_kernel//2)
    S = D_pool.flatten(1).max(dim=1)[0]  # (B,)
    return D, S

In [ ]:
class LGFDRModel(nn.Module):
        def __init__(self, swin_model=model_name_globally, pool_kernel=8, srh_latent=18, freeze_extractor=True): # Add freeze_extractor param
            super().__init__()
            self.extractor = SwinFeatureExtractor(model_name=swin_model, pool_kernel=pool_kernel)
            # We'll infer C after first forward pass
            self.local = None
            self.global_ = None
            self.srh = None
            self.srh_latent = srh_latent
            self.freeze_extractor = freeze_extractor # Store the flag

            if self.freeze_extractor:
                print(f"Freezing parameters of the feature extractor: {swin_model}")
                for param in self.extractor.parameters():
                    param.requires_grad = False

        def build_heads(self, C):
            # Standard: Build heads on first run
            self.local = LocalCAE(C)
            self.global_ = GlobalCAE(C)
            self.srh = SRH(C, latent_dim=self.srh_latent)
            # Move the newly created heads to the same device as the model
            self.local.to(next(self.parameters()).device)
            self.global_.to(next(self.parameters()).device)
            self.srh.to(next(self.parameters()).device)

        def forward(self, x, is_train=True):
            # Ensure extractor is in eval mode if frozen (important for BN layers)
            # Although SwinFeatureExtractor has freeze_bn logic,
            # putting the module itself in eval is safer if you fully freeze it.
            if self.freeze_extractor:
                 self.extractor.eval()
            else:
                 # If not frozen, its mode follows the parent LGFDRModel's mode (train/eval)
                 self.extractor.train(mode=is_train) # Ensure correct mode if not frozen


            E = self.extractor(x)

            if self.local is None:
                # Ensure C is inferred before building heads, even if extractor is frozen
                self.build_heads(E.shape[1])

            # Only inject noise if in training mode AND the extractor is not frozen,
            # or if you want noise on top of frozen features.
            # Common approach: noise on features regardless of freezing
            I = inject_noise(E, is_train) # Noise injection depends only on is_train flag


            L = self.local(I)
            G = self.global_(I)

            # Explicitly resize G to match L's spatial dimensions before passing to SRH
            if L.shape[-2:] != G.shape[-2:]:
                 G_aligned = F.interpolate(G, size=L.shape[-2:], mode='bilinear', align_corners=False)
            else:
                 G_aligned = G

            U = self.srh(L, G_aligned)

            return U, E

In [ ]:
# 7. Training and Inference Step Example (Pixel and Image-level Anomaly Scores)
import torch.optim as optim

# Example usage:
# model = LGFDRModel()
# optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_step(model, x):
    model.train()
    U, E = model(x, is_train=True)
    # L2 (MSE) loss per pixel, averaged
    loss = ((U - E) ** 2).mean()
    return loss

def inference_step(model, x):
    model.eval()
    with torch.no_grad():
        U, E = model(x, is_train=False)
        D_map, S_score = anomaly_maps(U, E, input_size=(images_size_globally, images_size_globally), pool_kernel=32)
    return D_map, S_score  # Pixel-level map, image-level score

In [ ]:
# ---------- Example usage and Debugging ----------

# --- Configuration for data loading ---
wood_dataset_root = f"{DATA_ROOT}/wood_panels/raw" # <<< CHANGE THIS TO YOUR DATASET ROOT

# Mode 1: All classes combined
wood_type_to_load = None      # None loads all wood types
# camera_angle_to_load = None # None loads all camera angles

# Mode 2: Single class and angle
# wood_type_to_load = "SKU_A"       # Specify a wood type (e.g., "Oak", "Pine")
camera_angle_to_load = "BOTTOM" # Specify a camera angle (e.g., "Angle1", "Angle2")

In [ ]:
# 8a. Custom Anomaly Dataset Loader and Setup for Zip Extraction and JSON Labels

import os
import glob
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
import shutil
from tqdm import tqdm # tqdm is already imported
import zipfile
import json
from collections import Counter


# --- Configuration for data loading ---
# Define the path to the zip file on DagsHub
zip_file_path = f"{DATA_ROOT}/wood_panels/Clean.zip"
# Define the local directory where the zip will be extracted
local_extracted_data_root = "work/extracted_wood_data"


# --- Step 1 & 2: Download and Extract Zip File ---
print(f"Checking if zip file exists at: {zip_file_path}")
if not os.path.exists(zip_file_path):
    raise FileNotFoundError(f"Zip file not found at: {zip_file_path}")
else:
    print("Zip file found.")

print(f"Checking if local extraction directory exists: {local_extracted_data_root}")
if not os.path.exists(local_extracted_data_root):
    print(f"Creating local extraction directory: {local_extracted_data_root}")
    os.makedirs(local_extracted_data_root)

print(f"Extracting zip file to {local_extracted_data_root}...")
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        # Extract only if the directory is empty or doesn't exist, to avoid re-extraction
        if not os.listdir(local_extracted_data_root):
             # Wrap the extraction process with tqdm for a progress bar
             for member in tqdm(zip_ref.infolist(), desc='Extracting zip file'):
                 try:
                      zip_ref.extract(member, local_extracted_data_root)
                 except zipfile.BadZipFile:
                      print(f"Warning: Could not extract {member.filename} - BadZipFile.")
                 except Exception as e:
                      print(f"Warning: Error extracting {member.filename}: {e}")

             print("Extraction complete.")
        else:
             print("Extraction directory is not empty. Skipping extraction.")

except FileNotFoundError:
    print(f"Error: Zip file not found at {zip_file_path}. Please check the path.")
    raise # Re-raise the error after printing the message
except zipfile.BadZipFile:
    print(f"Error: Could not open zip file at {zip_file_path}. It might be corrupted.")
    # You might want to add logic to delete the file and try again or raise a more critical error
    raise
except Exception as e:
    print(f"An unexpected error occurred during zip handling: {e}")
    raise


# --- Step 3: Modified Custom Anomaly Dataset Loader ---
class WoodAnomalyDataset(Dataset):
    def __init__(self, extracted_root, split='train', img_size=images_size_globally, normal_test_split_ratio=0.2, panel_color_filter=None):
        """
        Args:
            extracted_root (str): Root directory where the zip was extracted.
            split (str): Dataset split ('train' for normal training data, 'test' for all test data).
            img_size (int): Desired image size.
            normal_test_split_ratio (float): Ratio of normal images to use for the test set.
                                             Applied only when split is 'test' and loading normal data.
            panel_color_filter (str, optional): If provided, only load images with this panel color.
                                                Defaults to None (load all panel colors).
        """
        self.extracted_root = extracted_root
        self.split = split
        self.img_size = img_size
        self.normal_test_split_ratio = normal_test_split_ratio
        self.panel_color_filter = panel_color_filter


        self.transform = T.Compose([
            T.Resize((img_size, img_size)),
            T.ToTensor(),
        ])

        # Find all image files recursively in the extracted root directory
        # Use glob to find all image files with common extensions
        image_extensions = ['*.png', '*.jpg', '*.jpeg', '*.bmp']
        all_img_paths = []
        for ext in image_extensions:
             all_img_paths.extend(glob.glob(os.path.join(extracted_root, '**', ext), recursive=True))

        all_img_paths = sorted(all_img_paths)


        # Collect samples by pairing image paths with their corresponding JSON file paths
        all_samples = [] # List of (image_path, is_anomaly, panel_color) tuples
        panel_color_counts = Counter() # Initialize counter for panel colors

        # Wrap the sample filtering loop with tqdm
        for img_path in tqdm(all_img_paths, desc="Categorizing images and loading metadata"):
             base_name, _ = os.path.splitext(os.path.basename(img_path))
             # Assume JSON file is in the same directory as the image file
             json_path = os.path.join(os.path.dirname(img_path), base_name + '.json')

             if os.path.exists(json_path):
                 try:
                     with open(json_path, 'r') as f:
                         metadata = json.load(f)

                     is_anomaly = 1 if metadata.get('anomaly', False) else 0 # Get anomaly status, default to False
                     panel_color = metadata.get('panel_color', 'Unknown') # Get panel_color, default to 'Unknown'

                     # Apply panel color filter if specified
                     if self.panel_color_filter is None or panel_color == self.panel_color_filter:
                         # Store all relevant info with the sample
                         all_samples.append((img_path, is_anomaly, panel_color))

                         # Count panel colors only for the samples that were successfully added
                         panel_color_counts[panel_color] += 1

                 except json.JSONDecodeError:
                     print(f"Warning: Could not decode JSON from {json_path}. Skipping this pair.")
                 except Exception as e:
                     print(f"Warning: Error processing {json_path}: {e}. Skipping this pair.")
             else:
                 # Warning might be noisy, consider suppressing if many images lack JSON
                 # print(f"Warning: Corresponding JSON file not found for image {img_path} at {json_path}. Skipping this image.")
                 pass # Suppress warning


        # Print panel color counts
        print("\nPanel Color Distribution Across All Found Samples:")
        for color, count in panel_color_counts.items():
            print(f" - {color}: {count}")

        # Separate normal and anomaly samples based on the 'all_samples' list
        normal_samples = [s for s in all_samples if s[1] == 0]
        anomaly_samples = [s for s in all_samples if s[1] == 1]


        num_normal_images = len(normal_samples)
        indices = list(range(num_normal_images))
        np.random.seed(42) # Use a fixed seed for reproducibility
        np.random.shuffle(indices)


        if split == 'train':
            # Use the majority of normal data for training
            split_idx = int(np.floor(self.normal_test_split_ratio * num_normal_images))
            train_normal_indices = indices[split_idx:]
            # Samples list now contains (path, is_anomaly, panel_color) tuples
            self.samples = [normal_samples[i] for i in train_normal_indices]

        elif split == 'test':
            # Use a ratio of normal data + all anomaly data for testing
            split_idx = int(np.floor(self.normal_test_split_ratio * num_normal_images))
            test_normal_indices = indices[:split_idx]
            test_normal_samples = [normal_samples[i] for i in test_normal_indices]

            # Samples list now contains (path, is_anomaly, panel_color) tuples
            self.samples = test_normal_samples + anomaly_samples

        else:
            raise ValueError(f"Invalid split: {split}. Must be 'train' or 'test'.")

        print(f"Initialized {split} dataset with {len(self.samples)} samples.")
        if split == 'test':
            num_anomaly_test = sum(s[1] for s in self.samples)
            num_normal_test = len(self.samples) - num_anomaly_test
            print(f"Test set breakdown: {num_normal_test} normal, {num_anomaly_test} anomaly.")

        # Store panel color counts for potential external access or analysis
        self.panel_color_counts = panel_color_counts


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # Retrieve all info from the stored sample tuple
        img_path, is_anomaly, panel_color = self.samples[idx]

        try:
             img = Image.open(img_path).convert('RGB')
        except FileNotFoundError:
             print(f"Error: Image file not found at {img_path}. Skipping this sample.")
             # Handle this error gracefully, perhaps return a dummy or skip
             # For now, let's re-raise to avoid incorrect data in batch
             raise


        img = self.transform(img)

        # Always return a zero mask as pixel masks are not provided in the JSON
        mask = torch.zeros(1, self.img_size, self.img_size)

        # Return img, zero mask, original image path, and is_anomaly flag
        # Return panel_color as well
        return img, mask, img_path, is_anomaly, panel_color # Return anomaly status, path, AND panel_color


# ---------- Main Execution Block within cell 8a ----------

print(f"Setting up data loading using extracted data from: {local_extracted_data_root}")

# Instantiate datasets using the local extracted root
# No tqdm needed here, tqdm is inside the Dataset.__init__
richest_panel_color = "black" # <<< SET THE RICHEST PANEL COLOR FOR TRAINING HERE
print(f"\nInstantiating train dataset with filter for richest panel color: {richest_panel_color}")
train_dataset = WoodAnomalyDataset(local_extracted_data_root, split='train', img_size=images_size_globally, panel_color_filter=richest_panel_color)


# You can change "light_brown" to the panel color you want to test
test_panel_color = "black" # <<< SET THE PANEL COLOR FOR TESTING HERE (CHANGED TO NERO)
print(f"\nInstantiating test dataset with filter for panel color: {test_panel_color}")
test_dataset = WoodAnomalyDataset(local_extracted_data_root, split='test', img_size=images_size_globally, panel_color_filter=test_panel_color)


# Adjust batch size and num_workers as needed
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=os.cpu_count() // 2)
# Shuffle test loader to see a mix of normal and anomaly in samples, but keep batch size 1 for inference
# Updated the return values in the DataLoader to match the Dataset's __getitem__
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=os.cpu_count() // 2)


print("Datasets and dataloaders initialized.")
print(f"Train images: {len(train_dataset)}")
print(f"Test images: {len(test_dataset)}")

# Re-calculate and print the test set breakdown based on the actual test_dataset
num_anomaly_test = sum(s[1] for s in test_dataset.samples)
num_normal_test = len(test_dataset.samples) - num_anomaly_test
print(f"Test set: {num_normal_test} normal images, {num_anomaly_test} anomaly images.")

# Note: Data is now loaded directly from the local extracted directory.

Checking if zip file exists at: <DATA_ROOT>/wood_panels/Clean.zip
Zip file found.
Checking if local extraction directory exists: work/extracted_wood_data
Extracting zip file to work/extracted_wood_data...
Extraction directory is not empty. Skipping extraction.
Setting up data loading using extracted data from: work/extracted_wood_data

Instantiating train dataset with filter for richest panel color: black

Panel Color Distribution Across All Found Samples:
 - black: 6244
Initialized train dataset with 4665 samples.

Instantiating test dataset with filter for panel color: black

Panel Color Distribution Across All Found Samples:
 - black: 6244
Initialized test dataset with 1579 samples.
Test set breakdown: 1166 normal, 413 anomaly.
Datasets and dataloaders initialized.
Train images: 4665
Test images: 1579
Test set: 1166 normal images, 413 anomaly images.


In [ ]:
# 8b. Training, Testing Loop, and Evaluation with Checkpointing

from tqdm import tqdm
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
import torch
import torch.optim as optim
import torch.nn as nn
from torch.optim import lr_scheduler
import os # Import os for path joining

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Define checkpoint path
checkpoint_dir = f"{DATA_ROOT}/wood_panels/checkpointssingle"
os.makedirs(checkpoint_dir, exist_ok=True) # Create checkpoint directory if it doesn't exist
checkpoint_path = os.path.join(checkpoint_dir, "model_checkpoint.pth")
print(f"Checkpoint path: {checkpoint_path}")


# Instantiate the model. Start with the extractor frozen initially if no checkpoint is loaded.
# The freeze_extractor flag will be updated if a checkpoint is loaded.
model = LGFDRModel(freeze_extractor=True).to(device)

# --- Trigger build_heads before optimizer initialization ---
# Simulate a forward pass with a dummy tensor to initialize the heads
# Use a smaller dummy input if necessary to save memory
# Get shape from actual data, assuming train_loader is not empty
# Note: train_loader is initialized in cell 8a
input_size = next(iter(train_loader))[0].shape
dummy_input = torch.zeros(1, input_size[1], input_size[2], input_size[3]).to(device)
print("Triggering model heads initialization with a dummy input...")
with torch.no_grad():
    _ = model(dummy_input, is_train=False) # Use is_train=False for dummy pass
print("Model heads initialized.")
# --- End of trigger block ---

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params}")
print(f"Trainable parameters: {trainable_params}")


# --- Dynamic Training Setup ---
# Maximum total epochs to run (as a safeguard)
max_total_epochs = 500

# --- Patience-based Early Stopping and Unfreezing Variables ---
best_train_loss = float('inf')
patience_counter_stop = 0
patience_counter_unfreeze = 0
patience_value_stop = 10 # How many epochs to wait for IMPROVEMENT before stopping
patience_value_unfreeze = 30 # How many epochs to wait for IMPROVEMENT before unfreezing
min_delta = 0.00001 # Minimum improvement required to reset patience (for both unfreeze and stop)

base_lr = 1e-1 # Initial learning rate for heads
finetune_lr = 1e-4 # Lower learning rate for unfreeze phase

start_epoch = 0
extractor_frozen = True # Start assuming frozen, updated if checkpoint loaded


# Initialize optimizer for trainable parameters (initially only heads)
# This will be re-initialized if a checkpoint is loaded
trainable_params_freeze = filter(lambda p: p.requires_grad, model.parameters())
optimizer = torch.optim.Adam(trainable_params_freeze, lr=base_lr)

# Initialize the learning rate scheduler
# This will be re-initialized if a checkpoint is loaded
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, )


# --- Load Checkpoint if exists ---
if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from {checkpoint_path}")
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])

        # Re-initialize optimizer BEFORE loading state_dict to match parameters
        # Determine current learning rate based on whether extractor was frozen when saved
        saved_extractor_frozen_state = checkpoint.get('extractor_frozen', True) # Default to True for older checkpoints
        if saved_extractor_frozen_state:
             # Optimizer was for frozen parameters, use base_lr initially
             trainable_params_to_load = filter(lambda p: p.requires_grad, model.parameters())
             optimizer = torch.optim.Adam(trainable_params_to_load, lr=base_lr)
        else:
             # Optimizer was for unfrozen parameters, use finetune_lr initially
             # Temporarily unfreeze to get all parameters for optimizer init
             for param in model.extractor.parameters():
                  param.requires_grad = True
             trainable_params_to_load = filter(lambda p: p.requires_grad, model.parameters())
             optimizer = torch.optim.Adam(trainable_params_to_load, lr=finetune_lr)
             # Re-apply freezing based on loaded state
             if saved_extractor_frozen_state:
                  for param in model.extractor.parameters():
                       param.requires_grad = False


        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        # Re-initialize scheduler BEFORE loading state_dict
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, )
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])


        start_epoch = checkpoint['epoch'] + 1
        best_train_loss = checkpoint['best_train_loss']
        patience_counter_stop = checkpoint['patience_counter_stop']
        patience_counter_unfreeze = checkpoint['patience_counter_unfreeze']
        extractor_frozen = checkpoint['extractor_frozen'] # Load the extractor frozen state

        print(f"Checkpoint loaded. Resuming from epoch {start_epoch}")
        print(f"Loaded best_train_loss: {best_train_loss:.4f}")
        print(f"Loaded stop patience counter: {patience_counter_stop}")
        print(f"Loaded unfreeze patience counter: {patience_counter_unfreeze}")
        print(f"Loaded extractor frozen state: {extractor_frozen}")

    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        print("Starting training from scratch.")
        # Reset variables if loading failed
        start_epoch = 0
        best_train_loss = float('inf')
        patience_counter_stop = 0
        patience_counter_unfreeze = 0
        extractor_frozen = True # Ensure extractor is frozen if starting from scratch

        # Ensure optimizer and scheduler are initialized correctly for starting from scratch
        trainable_params_freeze = filter(lambda p: p.requires_grad, model.parameters())
        optimizer = torch.optim.Adam(trainable_params_freeze, lr=base_lr)
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, )


print(f"Starting dynamic training (max {max_total_epochs} epochs, unfreeze based on patience, stop based on patience).")

# Ensure initial trainable state matches the loaded or default frozen state
if extractor_frozen:
    print("Initial phase: Training with Extractor Frozen")
    # Ensure only non-extractor params require grad
    for name, param in model.named_parameters():
        if 'extractor' in name:
            param.requires_grad = False
        else:
             param.requires_grad = True
else:
     print("Initial phase: Training with Extractor Unfrozen")
     # Ensure all params require grad
     for param in model.parameters():
         param.requires_grad = True


for epoch in range(start_epoch, max_total_epochs):
    model.train()
    total_loss = 0

    # Print current learning rate
    current_lr = optimizer.param_groups[0]['lr']
    print(f"\nEpoch {epoch+1}/{max_total_epochs} | Current LR: {current_lr:.6f}")


    # Keep extractor in eval mode if frozen, set to train if unfrozen
    if extractor_frozen:
        model.extractor.eval()
    else:
        # When unfrozen, the extractor's mode follows the parent LGFDRModel's mode (train/eval)
        model.extractor.train() # Ensure correct mode if not frozen


    for imgs, masks, paths, is_anomaly, panel_color in tqdm(train_loader, desc=f"Epoch {epoch+1}/{max_total_epochs}"):
        imgs = imgs.to(device)
        # Masks, paths, is_anomaly, and panel_color are not needed for the train_step loss calculation
        loss = train_step(model, imgs)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)

    avg_train_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f}")

    # --- Step the scheduler based on training loss ---
    scheduler.step(avg_train_loss)

    # --- Patience and Early Stopping Logic ---
    # Check for improvement relative to the best loss seen so far
    if avg_train_loss < best_train_loss - min_delta:
        # Improvement found
        best_train_loss = avg_train_loss
        patience_counter_stop = 0 # Reset stop patience
        patience_counter_unfreeze = 0 # Reset unfreeze patience
        print(f"Epoch {epoch+1}: Loss improved to {avg_train_loss:.4f}. Patience counters reset.")
        # Optionally save the best model state here
        # torch.save(model.state_dict(), 'best_model.pth')
    else:
        # No significant improvement
        patience_counter_stop += 1
        if extractor_frozen:
            # Only increment unfreeze patience if extractor is still frozen
             patience_counter_unfreeze += 1
        print(f"Epoch {epoch+1}: No significant improvement. Stop patience: {patience_counter_stop}/{patience_value_stop}, Unfreeze patience: {patience_counter_unfreeze}/{patience_value_unfreeze}")


    # --- Check Patience Condition to Unfreeze ---
    # Unfreeze ONLY if patience for unfreezing is exhausted AND extractor is still frozen
    if extractor_frozen and patience_counter_unfreeze >= patience_value_unfreeze:
        print(f"\n--- Unfreeze patience ({patience_value_unfreeze}) exhausted. Unfreezing extractor parameters. ---")
        extractor_frozen = False
        model.freeze_extractor = False # Update the internal flag
        # --- Reset best_train_loss upon unfreezing ---
        best_train_loss = float('inf') # Reset best loss for the new phase
        print("Best train loss reset for unfrozen phase.")
        # No need to reset patience_counter_unfreeze here, as it will be reset when loss improves

        # Unfreeze the extractor parameters
        print("Unfreezing feature extractor parameters...")
        for name, param in model.extractor.named_parameters():
             if not param.requires_grad: # Only print if it was actually frozen
                 param.requires_grad = True
                 # print(f" - Unfrozen: {name}")


        # Re-initialize the optimizer to include the newly unfrozen parameters
        # AND re-initialize the scheduler with the new optimizer
        trainable_params_unfreeze = filter(lambda p: p.requires_grad, model.parameters())
        num_trainable_unfreeze = sum(1 for _ in trainable_params_unfreeze)
        print(f"Number of trainable parameters after unfreezing: {num_trainable_unfreeze}")
        # Re-create iterator for optimizer
        trainable_params_unfreeze = filter(lambda p: p.requires_grad, model.parameters()) # Need to re-create iterator

        optimizer = torch.optim.Adam(trainable_params_unfreeze, lr=finetune_lr)
        print(f"Optimizer re-initialized with learning rate: {finetune_lr}")

        # Re-initialize the scheduler with the NEW optimizer
        scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, ) # DECREASED patience
        print("Scheduler re-initialized with new optimizer.")


        # You might want to print a message indicating the phase has changed
        print("--- Phase: Training with Extractor Unfrozen ---")


    # --- Check Patience Condition to Stop Training ---
    # Stop ONLY if patience for stopping is exhausted
    if patience_counter_stop >= patience_value_stop:
         print(f"\n--- Stop patience ({patience_value_stop}) exhausted. Training loss did not improve significantly for {patience_value_stop} epochs. Stopping training. ---")
         break # Exit the main epoch loop


    # --- Save Checkpoint ---
    print(f"Saving checkpoint after epoch {epoch+1} to {checkpoint_path}")
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_train_loss': best_train_loss,
        'patience_counter_stop': patience_counter_stop,
        'patience_counter_unfreeze': patience_counter_unfreeze,
        'extractor_frozen': extractor_frozen # Save the extractor frozen state
    }, checkpoint_path)


print("\n--- Training Complete ---")

# --------------- Testing and Evaluation ---------------

model.eval()
all_scores = []
all_labels = [] # This will now store the is_anomaly flag
all_pixel_scores = []
all_pixel_labels = [] # This will still be all zeros

with torch.no_grad():
    for imgs, masks, paths, is_anomaly, panel_color in tqdm(test_loader, desc="Testing"):
        imgs = imgs.to(device)
        # Ensure extractor is in eval mode during inference
        model.extractor.eval()
        # Pass is_train=False during inference step
        U, E = model(imgs, is_train=False) # Use the model's forward
        D_map, S_score = anomaly_maps(U, E, input_size=(images_size_globally, images_size_globally), pool_kernel=32)


        # For image-level AUROC: use the is_anomaly flag from the dataset
        all_scores.append(S_score.cpu().item())
        all_labels.append(is_anomaly.cpu().item()) # Append the is_anomaly flag
        # For pixel-level AUROC/AP
        all_pixel_scores.append(D_map.cpu().numpy().flatten())
        # Since we don't have ground truth pixel masks, pixel-level metrics are not valid.
        # We'll append the zero mask flattened, but note that evaluation on this will be useless.
        all_pixel_labels.append(masks.cpu().numpy().flatten())


# Compute metrics
all_scores = np.array(all_scores)
all_labels = np.array(all_labels)
print("\n--- Evaluation Results ---")
print("Unique image labels in test set:", set(all_labels))
print("Number of test images:", len(all_labels))
if np.unique(all_labels).size > 1:
    print("Image-level AUROC:", roc_auc_score(all_labels, all_scores))
else:
    # This will happen if the test set only contains normal or only anomaly images
    print("Image-level AUROC: undefined (only one class present in test set)")

if all_pixel_labels:
    px_scores = np.concatenate(all_pixel_scores)
    px_labels = np.concatenate(all_pixel_labels)
    # Since px_labels will be all zeros (no masks), pixel-level metrics are not valid.
    # We can still calculate them, but the result will be misleading.
    # Only calculate if there's actual variation in labels (which there won't be with zero masks)
    # and if there are scores (which there will be).
    # Let's add a check if there are any positive labels before calculating.
    if np.sum(px_labels) > 0: # This condition will likely never be met with zero masks
        print("Pixel-level AUROC:", roc_auc_score(px_labels, px_scores))
        print("Pixel-level AP:", average_precision_score(px_labels, px_scores))
    else:
         print("Pixel-level AUROC/AP: undefined (no positive mask pixels provided in dataset)")

else:
    print("No pixel-level masks found.")

# ------------ Instructions for Dataset Setup ------------
# Ensure your wood dataset is structured as:
# root_folder/
# ├── wood_type_1/
# │   ├── camera_angle_A/
# │   │   ├── normal_image1.bmp/png/jpg
# │   │   ├── normal_image2.bmp/png/jpg
# │   │   └── anomaly/
# │   │       ├── anomaly_image1.bmp/png/jpg
# │   │       └── anomaly_image2.bmp/png/jpg
# │   ├── camera_angle_B/
# │   │   ├── normal_imageX.bmp/png/jpg
# │   │   └── anomaly/
# │   │       └── anomaly_imageY.bmp/png/jpg
# └── wood_type_2/
#     ├── camera_angle_A/
#     ... etc.
# Set wood_dataset_root to the path of root_folder.
# Use wood_type_to_load and camera_angle_to_load = None for all data, or specify for a single subset.

Checkpoint path: <DATA_ROOT>/wood_panels/checkpointssingle/model_checkpoint.pth
Freezing parameters of the feature extractor: swin_base_patch4_window7_224
Triggering model heads initialization with a dummy input...
Model heads initialized.
Total parameters: 95626186
Trainable parameters: 8885010

Trainable Parameter Names and Shapes:
 - local.encoder.0.conv.weight: torch.Size([384, 384, 1, 1]), Num elements: 147456
 - local.encoder.0.bn.weight: torch.Size([384]), Num elements: 384
 - local.encoder.0.bn.bias: torch.Size([384]), Num elements: 384
 - local.encoder.1.conv.weight: torch.Size([384, 384, 1, 1]), Num elements: 147456
... [39 lines of log omitted] ...
Loaded stop patience counter: 9
Loaded unfreeze patience counter: 9
Loaded extractor frozen state: True
Starting dynamic training (max 500 epochs, unfreeze based on patience, stop based on patience).
Initial phase: Training with Extractor Frozen

Epoch 134/500 | Current LR: 0.006250
Epoch 134 | Train Loss: 0.0029
Epoch 134: No sig

In [ ]:
# 10. Visualize Original Image, Feature Map E, Reconstructed Feature Map U, Raw Diff, Upsampled Anomaly Map


import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm
from collections import defaultdict # Import defaultdict

# Define the main output directory for saving visualizations (This is not used for plotting, but kept for consistency if needed later)
output_dir = f"{DATA_ROOT}/wood_panels/resultssingle"

# Define checkpoint path (should match the path used in the training cell)
checkpoint_dir = f"{DATA_ROOT}/wood_panels/checkpointssingle"
checkpoint_path = os.path.join(checkpoint_dir, "model_checkpoint.pth")
print(f"Looking for checkpoint at: {checkpoint_path}")

# Instantiate the model (this is needed before loading state_dict)
# We don't need to freeze the extractor here for inference/visualization
# Set freeze_extractor to False for inference
model = LGFDRModel(freeze_extractor=False).to(device)

# --- Trigger build_heads before loading checkpoint ---
# Simulate a forward pass with a dummy tensor to initialize the heads
# Get shape from actual data, assuming test_loader is not empty
# Check if test_loader has any data first
try:
    input_size = next(iter(test_loader))[0][0].shape # Get shape from the first image in the first batch
    dummy_input = torch.zeros(1, input_size[0], input_size[1], input_size[2]).to(device)
    print("Triggering model heads initialization with a dummy input before loading checkpoint...")
    with torch.no_grad():
        _ = model(dummy_input, is_train=False) # Use is_train=False for dummy pass
    print("Model heads initialized.")
except StopIteration:
    print("Warning: Test loader is empty. Cannot trigger model heads initialization.")
    # Handle the case where test_loader is empty - perhaps skip visualization or show an error
    # For now, we'll proceed, but model heads won't be built until the first real batch if data appears.
    # This might cause issues later if data is only available after this cell.


# --- Load the latest Checkpoint if exists ---
if os.path.exists(checkpoint_path):
    print(f"Loading model state from checkpoint: {checkpoint_path}")
    try:
        # Load checkpoint without explicitly specifying epoch
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print("Model state loaded successfully.")
        # You can optionally print the epoch from the loaded checkpoint
        loaded_epoch = checkpoint.get('epoch', 'N/A')
        print(f"Checkpoint saved at epoch: {loaded_epoch}")


    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        print("Proceeding with potentially untrained model.")
else:
    print("No checkpoint found. Proceeding with untrained model.")


# Ensure the model is in evaluation mode
model.eval()

# --- Collect Anomaly Samples and Group by Panel Color ---
print("\nCollecting anomaly samples and grouping by panel color...")
anomaly_samples_by_color = defaultdict(list)

if len(test_loader) > 0:
    with torch.no_grad():
        # Iterate through the entire test loader
        # Unpack all 5 values returned by the DataLoader
        for imgs, masks, paths, is_anomaly, panel_color in tqdm(test_loader, desc="Collecting anomaly samples"):
            # Check if the current image is an anomaly
            if is_anomaly.cpu().item() > 0:
                # Store the data for later processing
                anomaly_samples_by_color[panel_color[0]].append((imgs.cpu(), masks.cpu(), paths[0], is_anomaly.cpu().item(), panel_color[0]))
else:
    print("Test loader is empty. Skipping visualization.")


# --- Select a maximum of 5 random anomaly samples per panel color ---
selected_anomaly_samples = []
num_samples_per_color = 5 # Maximum number of samples to visualize per color

print(f"\nSelecting up to {num_samples_per_color} random anomaly samples per panel color...")
np.random.seed(42) # Ensure reproducibility of random selection

for color, samples in anomaly_samples_by_color.items():
    if len(samples) > num_samples_per_color:
        # Randomly select num_samples_per_color indices
        selected_indices = np.random.choice(len(samples), num_samples_per_color, replace=False)
        selected_samples_for_color = [samples[i] for i in selected_indices]
    else:
        # Take all samples if fewer than num_samples_per_color
        selected_samples_for_color = samples

    selected_anomaly_samples.extend(selected_samples_for_color)
    print(f" - {color}: Selected {len(selected_samples_for_color)} out of {len(samples)} anomaly samples.")

print(f"\nTotal anomaly samples selected for visualization: {len(selected_anomaly_samples)}")


# --- Calculate Global Min and Max for Heatmaps Across Selected Anomaly Samples ---
# This ensures a consistent color scale ONLY for the images being visualized.
print("\nCalculating global min/max for anomaly heatmaps across the selected anomaly samples...")
all_selected_anomaly_raw_diff_maps_flat = []
all_selected_anomaly_upsampled_anomaly_maps_flat = []

if selected_anomaly_samples:
    with torch.no_grad():
        # Iterate through the selected anomaly samples
        for imgs, masks, path, is_anomaly_single, panel_color_single in tqdm(selected_anomaly_samples, desc="Calculating global scale on selected samples"):
                img = imgs.to(device)

                # Ensure heads are built if this is the very first inference after init
                if model.local is None:
                     dummy_input_for_build = img if img.shape[0] == 1 else img[0].unsqueeze(0)
                     model.build_heads(model.extractor(dummy_input_for_build).shape[1])


                # Get extracted and reconstructed features
                U, E = model(img, is_train=False)

                # Compute raw diff and upsampled anomaly map
                d_raw = torch.norm(U - E, p=2, dim=1)
                D_map_upsampled, _ = anomaly_maps(U, E, input_size=img.shape[-2:], pool_kernel=32)

                # Append flattened numpy arrays
                all_selected_anomaly_raw_diff_maps_flat.append(d_raw.cpu().numpy().flatten())
                all_selected_anomaly_upsampled_anomaly_maps_flat.append(D_map_upsampled.cpu().numpy().flatten())

else:
    print("No anomaly samples selected for visualization. Skipping global scale calculation.")

# Concatenate all flattened arrays and find global min/max
if all_selected_anomaly_raw_diff_maps_flat:
    global_raw_diff_min = np.min(np.concatenate(all_selected_anomaly_raw_diff_maps_flat))
    global_raw_diff_max = np.max(np.concatenate(all_selected_anomaly_raw_diff_maps_flat))
else:
    global_raw_diff_min, global_raw_diff_max = 0, 1 # Default if no anomaly data

if all_selected_anomaly_upsampled_anomaly_maps_flat:
    global_upsampled_min = np.min(np.concatenate(all_selected_anomaly_upsampled_anomaly_maps_flat))
    global_upsampled_max = np.max(np.concatenate(all_selected_anomaly_upsampled_anomaly_maps_flat))
else:
    global_upsampled_min, global_upsampled_max = 0, 1 # Default if no anomaly data


print(f"Global Raw Diff Min (Selected Anomaly Samples): {global_raw_diff_min:.4f}, Max: {global_raw_diff_max:.4f}")
print(f"Global Upsampled Anomaly Min (Selected Anomaly Samples): {global_upsampled_min:.4f}, Max: {global_upsampled_max:.4f}")


# --- Generate and Display Plots for the Selected Anomaly Test Images, Organized by Panel Color ---
print("\nGenerating and displaying visualizations for SELECTED ANOMALY test images...")
if selected_anomaly_samples:
    with torch.no_grad():
        # Iterate through the selected anomaly samples data
        for imgs, masks, path, is_anomaly_single, panel_color_single in tqdm(selected_anomaly_samples, desc="Displaying selected anomaly visualizations"):
                img = imgs.to(device)

                # Ensure heads are built if this is the very first inference after init
                if model.local is None:
                     dummy_input_for_build = img if img.shape[0] == 1 else img[0].unsqueeze(0)
                     model.build_heads(model.extractor(dummy_input_for_build).shape[1])


                # Get extracted and reconstructed features
                U, E = model(img, is_train=False)

                # Compute raw diff and upsampled anomaly map
                d_raw = torch.norm(U - E, p=2, dim=1).squeeze(0).cpu().numpy() # (H_feat, W_feat)
                D_map_upsampled, S_score = anomaly_maps(U, E, input_size=img.shape[-2:], pool_kernel=32)
                D_map_upsampled_np = D_map_upsampled.squeeze(0).cpu().numpy() # (H, W)

                # Prepare other numpy arrays for plotting
                original_img_np = img.squeeze(0).cpu().permute(1, 2, 0).numpy() # (H, W, C)
                E_avg_np = E.squeeze(0).mean(dim=0).cpu().numpy() # (H_feat, W_feat)
                U_avg_np = U.squeeze(0).mean(dim=0).cpu().numpy() # (H_feat, W_feat)

                # Create the figure and plot
                fig, axes = plt.subplots(1, 5, figsize=(25, 5))

                # Plot Original Image
                axes[0].imshow(np.clip(original_img_np, 0, 1))
                axes[0].set_title(f"Anomaly Image\n({os.path.basename(path)})")
                axes[0].axis('off')

                # Plot Feature Map E
                axes[1].imshow(E_avg_np, cmap='gray')
                axes[1].set_title("Feature Map E\n(Avg over channels)")
                axes[1].axis('off')

                # Plot Reconstructed Feature Map U
                axes[2].imshow(U_avg_np, cmap='gray')
                axes[2].set_title("Reconstructed Feature Map U\n(Avg over channels)")
                axes[2].axis('off')

                # Plot Raw Diff (||U-E||) with Global Scale (calculated from selected anomaly samples)
                im4 = axes[3].imshow(d_raw, cmap='hot', vmin=global_raw_diff_min, vmax=global_raw_diff_max)
                axes[3].set_title("Raw Diff (||U-E||)\n(Feature Map Size - Selected Anomaly Scale)")
                axes[3].axis('off')
                fig.colorbar(im4, ax=axes[3], label='L2 Norm')

                # Plot Upsampled Anomaly Heatmap with Global Scale (calculated from selected anomaly samples)
                im5 = axes[4].imshow(D_map_upsampled_np, cmap='hot', vmin=global_upsampled_min, vmax=global_upsampled_max)
                axes[4].set_title(f"Upsampled Anomaly Heatmap\nScore: {S_score.cpu().item():.4f} (Selected Anomaly Scale)")
                axes[4].axis('off')
                fig.colorbar(im5, ax=axes[4], label='Anomaly Score')

                plt.tight_layout()

                # Display the figure
                plt.show()

                # Close the figure to free up memory after displaying
                plt.close(fig)
else:
    print("No anomaly samples selected for visualization. Skipping visualization.")


print("Visualization complete.")

Looking for checkpoint at: <DATA_ROOT>/wood_panels/checkpointssingle/model_checkpoint.pth
Triggering model heads initialization with a dummy input before loading checkpoint...
Model heads initialized.
Loading model state from checkpoint: <DATA_ROOT>/wood_panels/checkpointssingle/model_checkpoint.pth
Model state loaded successfully.
Checkpoint saved at epoch: 132


Selecting up to 5 random anomaly samples per panel color...
 - black: Selected 5 out of 413 anomaly samples.

Total anomaly samples selected for visualization: 5

Calculating global min/max for anomaly heatmaps across the selected anomaly samples...
Global Raw Diff Min (Selected Anomaly Samples): 0.2122, Max: 4.5226
Global Upsampled Anomaly Min (Selected Anomaly Samples): 0.2128, Max: 4.5085

Generating and displaying visualizations for SELECTED ANOMALY test images...

Visualization complete.


In [ ]:
# 11. Save Visualizations for Anomaly Test Images Only, Organized by Panel Color

import os
import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm import tqdm

# Define the main output directory for saving visualizations
output_dir = f"{DATA_ROOT}/wood_panels/resultssingle2"

# Create the main output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)
print(f"Main output directory for visualizations: {output_dir}")

# Define checkpoint path (should match the path used in the training cell)
checkpoint_dir = f"{DATA_ROOT}/wood_panels/checkpointssingle"
checkpoint_path = os.path.join(checkpoint_dir, "model_checkpoint.pth")
print(f"Looking for checkpoint at: {checkpoint_path}")

# Instantiate the model (this is needed before loading state_dict)
# We don't need to freeze the extractor here for inference/visualization
model = LGFDRModel(freeze_extractor=False).to(device) # Instantiate model

# --- Trigger build_heads before loading checkpoint ---
# Simulate a forward pass with a dummy tensor to initialize the heads
# Get shape from actual data, assuming test_loader is not empty
input_size = next(iter(test_loader))[0][0].shape # Get shape from the first image in the first batch
dummy_input = torch.zeros(1, input_size[0], input_size[1], input_size[2]).to(device)
print("Triggering model heads initialization with a dummy input before loading checkpoint...")
with torch.no_grad():
    _ = model(dummy_input, is_train=False) # Use is_train=False for dummy pass
print("Model heads initialized.")
# --- End of trigger block ---


# --- Load Checkpoint if exists ---
if os.path.exists(checkpoint_path):
    print(f"Loading model state from checkpoint: {checkpoint_path}")
    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print("Model state loaded successfully.")
        # You can optionally load other states like epoch, best_loss, etc., but not strictly needed for inference.
        # start_epoch = checkpoint['epoch'] + 1
        # best_train_loss = checkpoint['best_train_loss']
        # patience_counter_stop = checkpoint['patience_counter_stop']
        # patience_counter_unfreeze = checkpoint['patience_counter_unfreeze']
        # extractor_frozen = checkpoint['extractor_frozen'] # Load the extractor frozen state

    except Exception as e:
        print(f"Error loading checkpoint: {e}")
        print("Proceeding with potentially untrained model.")
else:
    print("No checkpoint found. Proceeding with untrained model.")


# Ensure the model is in evaluation mode
model.eval()

# --- Calculate Global Min and Max for Heatmaps Across ALL Test Set (for consistent scale) ---
# It's often best to calculate the global min/max across the entire test set
# (including normal and anomaly) to ensure a consistent color scale for comparisons.
# If you prefer the scale to be based *only* on anomaly images, you would need
# to first identify all anomaly images and then calculate min/max only from them.
# Let's stick to calculating across the full test set for a generally better visual comparison.
print("Calculating global min/max for anomaly heatmaps across the entire test set...")
all_upsampled_anomaly_maps_flat = []
all_raw_diff_maps_flat = []

with torch.no_grad():
    # Iterate through the entire test loader to collect anomaly map values
    # Unpack all 5 values returned by the DataLoader
    for imgs, masks, paths, is_anomaly, panel_color in tqdm(test_loader, desc="Collecting data for global scale"):
        imgs = imgs.to(device)

        # Ensure heads are built if this is the very first inference after init
        # This check might be redundant if loading from checkpoint, but safe.
        if model.local is None:
             dummy_input_for_build = imgs if imgs.shape[0] == 1 else imgs[0].unsqueeze(0)
             model.build_heads(model.extractor(dummy_input_for_build).shape[1])


        # Get extracted and reconstructed features
        U, E = model(imgs, is_train=False)

        # Compute raw diff and upsampled anomaly map
        d_raw = torch.norm(U - E, p=2, dim=1)
        D_map_upsampled, _ = anomaly_maps(U, E, input_size=imgs.shape[-2:], pool_kernel=32)

        # Append flattened numpy arrays
        all_raw_diff_maps_flat.append(d_raw.cpu().numpy().flatten())
        all_upsampled_anomaly_maps_flat.append(D_map_upsampled.cpu().numpy().flatten())

# Concatenate all flattened arrays and find global min/max
if all_raw_diff_maps_flat:
    global_raw_diff_min = np.min(np.concatenate(all_raw_diff_maps_flat))
    global_raw_diff_max = np.max(np.concatenate(all_raw_diff_maps_flat))
else:
    global_raw_diff_min, global_raw_diff_max = 0, 1 # Default if no data

if all_upsampled_anomaly_maps_flat:
    global_upsampled_min = np.min(np.concatenate(all_upsampled_anomaly_maps_flat))
    global_upsampled_max = np.max(np.concatenate(all_upsampled_anomaly_maps_flat))
else:
    global_upsampled_min, global_upsampled_max = 0, 1 # Default if no data


print(f"Global Raw Diff Min: {global_raw_diff_min:.4f}, Max: {global_raw_diff_max:.4f}")
print(f"Global Upsampled Anomaly Min: {global_upsampled_min:.4f}, Max: {global_upsampled_max:.4f}")


# --- Generate and Save Plots for Each ANOMALY Test Image, Organized by Panel Color ---
print("\nGenerating and saving visualizations for ANOMALY test images, organized by panel color...")

with torch.no_grad():
    # Iterate through the entire test loader
    # Unpack all 5 values returned by the DataLoader
    for imgs, masks, paths, is_anomaly, panel_color in tqdm(test_loader, desc="Saving anomaly visualizations"):
        # Check if the current image is an anomaly
        if is_anomaly.cpu().item() > 0:
            img = imgs.to(device)
            path = paths[0] # Get the single path from the batch
            is_anomaly_single = is_anomaly.cpu().item() # Get the single anomaly status
            panel_color_single = panel_color[0] # Get the single panel color from the batch

            # Create panel color subdirectory if it doesn't exist
            panel_color_output_dir = os.path.join(output_dir, panel_color_single)
            os.makedirs(panel_color_output_dir, exist_ok=True)

            # Get extracted and reconstructed features
            U, E = model(img, is_train=False)

            # Compute raw diff and upsampled anomaly map
            d_raw = torch.norm(U - E, p=2, dim=1).squeeze(0).cpu().numpy() # (H_feat, W_feat)
            D_map_upsampled, S_score = anomaly_maps(U, E, input_size=img.shape[-2:], pool_kernel=32)
            D_map_upsampled_np = D_map_upsampled.squeeze(0).cpu().numpy() # (H, W)

            # Prepare other numpy arrays for plotting
            original_img_np = img.squeeze(0).cpu().permute(1, 2, 0).numpy() # (H, W, C)
            E_avg_np = E.squeeze(0).mean(dim=0).cpu().numpy() # (H_feat, W_feat)
            U_avg_np = U.squeeze(0).mean(dim=0).cpu().numpy() # (H_feat, W_feat)

            # Determine filename based on original image name and anomaly status
            base_name = os.path.splitext(os.path.basename(path))[0]
            anomaly_status_str = "Anomaly" # We are only saving anomaly images here
            filename = f"{base_name}_{anomaly_status_str}_Score_{S_score.cpu().item():.4f}.png"
            save_path = os.path.join(panel_color_output_dir, filename) # Save to panel color subdirectory

            # Create the figure and plot
            fig, axes = plt.subplots(1, 5, figsize=(25, 5))

            # Plot Original Image
            axes[0].imshow(np.clip(original_img_np, 0, 1))
            axes[0].set_title(f"{anomaly_status_str} Image\n({os.path.basename(path)})")
            axes[0].axis('off')

            # Plot Feature Map E
            axes[1].imshow(E_avg_np, cmap='gray')
            axes[1].set_title("Feature Map E\n(Avg over channels)")
            axes[1].axis('off')

            # Plot Reconstructed Feature Map U
            axes[2].imshow(U_avg_np, cmap='gray')
            axes[2].set_title("Reconstructed Feature Map U\n(Avg over channels)")
            axes[2].axis('off')

            # Plot Raw Diff (||U-E||) with Global Scale
            im4 = axes[3].imshow(d_raw, cmap='hot', vmin=global_raw_diff_min, vmax=global_raw_diff_max)
            axes[3].set_title("Raw Diff (||U-E||)\n(Feature Map Size - Global Scale)")
            axes[3].axis('off')
            fig.colorbar(im4, ax=axes[3], label='L2 Norm')

            # Plot Upsampled Anomaly Heatmap with Global Scale
            im5 = axes[4].imshow(D_map_upsampled_np, cmap='hot', vmin=global_upsampled_min, vmax=global_upsampled_max)
            axes[4].set_title(f"Upsampled Anomaly Heatmap\nScore: {S_score.cpu().item():.4f} (Global Scale)")
            axes[4].axis('off')
            fig.colorbar(im5, ax=axes[4], label='Anomaly Score')

            plt.tight_layout()

            # Save the figure
            plt.savefig(save_path)
            plt.close(fig) # Close the figure to free up memory

print("Visualization saving complete.")

Main output directory for visualizations: <DATA_ROOT>/wood_panels/resultssingle2
Looking for checkpoint at: <DATA_ROOT>/wood_panels/checkpointssingle/model_checkpoint.pth
Triggering model heads initialization with a dummy input before loading checkpoint...
Model heads initialized.
Loading model state from checkpoint: <DATA_ROOT>/wood_panels/checkpointssingle/model_checkpoint.pth
Model state loaded successfully.
Calculating global min/max for anomaly heatmaps across the entire test set...
Global Raw Diff Min: 0.1948, Max: 6.0538
Global Upsampled Anomaly Min: 0.1948, Max: 6.0283

Generating and saving visualizations for ANOMALY test images, organized by panel color...
Visualization saving complete.
